**STEP 1: Simulation**

In [1]:
import numpy as np
from scipy.stats import norm
def simulate_matrix(n, d, density, out_fra, corr):
    # Generate a simulated large-scale matrix ~ N(0,1)
    X = np.random.normal(loc=0, scale=1, size=(n, d))

    # Apply corr structure with normal distribution
                                       
    # Set the correlation strength to be the desired value
    corr_strength = corr
    # scaling the values in X with strength
    XX = X * corr_strength
    # Add noise to the correlation values
    XX = XX + np.random.normal(0,1-corr_strength**2,size=(n,d))
    # transform X to be correlated
    X = norm.cdf(XX)

    # Apply nonlinear by using ReLU Function
    X = np.maximum(0,X)

    # Add outliers by randomly selecting rows and adding 10 times entities ~ N(0,1)

    ### size of outliers
    size = int(out_fra * n)
    outlier_idx = np.random.choice(n, size=size, replace=False)
    X[outlier_idx] += 39 * np.random.normal(loc=0, scale=1, size=(size, d))

    # To sparse matrix by setting the desired fraction to be zero (out of density)
    X[np.random.rand(n, d) > density] = 0

    return X

# Matrix Settled
n = 5000       # number of rows         
d = 100        # number of columns
density = 0.87  # sparse density to make more 0 entities
out_fra = 0.2  # outliers' fraction
corr = 0.15    # correlation strength

X = simulate_matrix(n, d, density, out_fra, corr) # A matrix
Y = np.random.normal(loc=0,scale=1,size=(n,1)) # b vector
print("The simulated matrix is:")
print(np.array(X))
print("The number of 0 entities' ratio is:",int((X==0).sum())/(n*d))

The simulated matrix is:
[[0.93508477 0.70395204 0.56148154 ... 0.71773549 0.57689462 0.98776293]
 [0.04160173 0.66052154 0.76821671 ... 0.34965757 0.43490819 0.25245433]
 [0.62828296 0.30156543 0.13426031 ... 0.19734449 0.         0.63305826]
 ...
 [0.9970555  0.45977465 0.41427972 ... 0.         0.11252702 0.63970501]
 [0.24905556 0.         0.83170167 ... 0.40038649 0.58494097 0.36327915]
 [0.61631743 0.         0.72980576 ... 0.94529618 0.36160912 0.52764552]]
The number of 0 entities' ratio is: 0.130448


*Define functions for calculating ratio and norms*

In [2]:
def calculate_the_norm_square(A,b,x_selected):
    return (np.linalg.norm(A @ x_selected - b)) ** 2
def abs_error_ratio(A,b,x_hat_bar,x_star):
    return calculate_the_norm_square(A,b,x_hat_bar) - calculate_the_norm_square(A,b,x_star) / calculate_the_norm_square(A,b,x_star)

*Try to figure out what x_star should be exiquisitely*

In [3]:
from scipy.sparse import csr_matrix
from scipy.sparse.linalg import lsqr
from scipy.linalg import cho_factor, cho_solve
def solver(A,b,solver):
    if solver == 'lsqr':
    # lsqr algorithm
        X = csr_matrix(A)
        Y = np.array(b)
        x_star = lsqr(X, Y)[0]
    # cholesky decomposition algorithm
    elif solver == 'cholesky':
        L,low = cho_factor(A.T@A)
        x_star = cho_solve((L,low),A.T@b)
    # Direct solver
    elif solver == 'direct':
        x_star = np.linalg.inv(A.T@A)@A.T@b

    return x_star
x_star_lsqr = solver(X,Y,'lsqr')
x_star_cholesky = solver(X,Y,'cholesky')
x_star_direct = solver(X,Y,'direct')
norm_lsqr = calculate_the_norm_square(X,Y,x_star_lsqr)
norm_cholesky = calculate_the_norm_square(X,Y,x_star_cholesky)
norm_direct = calculate_the_norm_square(X,Y,x_star_direct)
print("The norm square of x_star calculated as by lsqr, cholesky and direct solver respectively are:")
print(norm_lsqr, norm_cholesky, norm_direct)
minimum_norm_square = min(norm_lsqr, norm_cholesky, norm_direct)
print("Here we output the minimum of the norm suqare for x_star calculated as:")
print(minimum_norm_square)

The norm square of x_star calculated as by lsqr, cholesky and direct solver respectively are:
25896912.36219433 4960.483833100384 4960.483833100382
Here we output the minimum of the norm suqare for x_star calculated as:
4960.483833100382


**STEP 2: Uniform Sampling sketch**

In [4]:
def uniform_sampling(A,b,n,m):
    

_IncompleteInputError: incomplete input (1783957440.py, line 2)